# Imports and Load Data

In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_colwidth', None)

In [2]:
games=pd.read_csv('data/games.csv.gz')
players=pd.read_csv('data/players.csv.gz')
stints=pd.read_csv('data/stints.csv.gz')
teams=pd.read_csv('data/teams.csv.gz')

# Data Cleaning

In [3]:
games['game_date']=pd.to_datetime(games['game_date'])

games.head()

,game_id,game_date,hosting_team_id,is_neutral,home_team_id,away_team_id,home_score,away_score,is_post_season,in_conference
0,2742787,2025-10-10,0,False,103331,103659,79.0,74.0,False,False
1,2721497,2025-10-11,0,False,103661,103369,78.0,83.0,False,False
2,2753208,2025-10-11,0,False,104311,103642,94.0,90.0,False,False
3,2739384,2025-10-12,0,False,104353,103297,72.0,68.0,False,False
4,2756992,2025-10-12,0,False,104135,104378,64.0,74.0,False,False


In [4]:
stints.head(5)

,game_id,stint_number,period,team_1_id,team_2_id,players_team_1,players_team_2,pts_team_1,pts_team_2,poss_team_1,...,start_secs_remaining,end_secs_remaining,fga_team_1,fga_team_2,fta_team_1,fta_team_2,oreb_team_1,oreb_team_2,tov_team_1,tov_team_2
0,2682853,1,1,103384,103384,1908702;2134479;2292664;1706626;2319448,2329174;2496768;2160639;2496771;2118863,7,6,8.0,...,2400,2142,7,7,0,0,0,0,1,1
1,2682853,2,1,103384,103384,2134479;2292664;1706626;2319448,2329174;2496768;2160639;2496771;2118863,0,0,0.0,...,2142,2142,0,0,0,0,0,0,0,0
2,2682853,3,1,103384,103384,2134479;2292664;2319448,2329174;2496768;2160639;2496771;2118863,0,0,0.0,...,2142,2142,0,0,0,0,0,0,0,0
3,2682853,4,1,103384,103384,2134479;2292664,2329174;2496768;2160639;2496771;2118863,0,0,0.0,...,2142,2142,0,0,0,0,0,0,0,0
4,2682853,5,1,103384,103384,2134479;2292664;2147206,2329174;2496768;2160639;2496771;2118863,0,0,0.0,...,2142,2142,0,0,0,0,0,0,0,0


In [5]:
stints.columns

Index(['game_id', 'stint_number', 'period', 'team_1_id', 'team_2_id',
       'players_team_1', 'players_team_2', 'pts_team_1', 'pts_team_2',
       'poss_team_1', 'poss_team_2', 'start_score_1', 'start_score_2',
       'end_score_1', 'end_score_2', 'start_secs_remaining',
       'end_secs_remaining', 'fga_team_1', 'fga_team_2', 'fta_team_1',
       'fta_team_2', 'oreb_team_1', 'oreb_team_2', 'tov_team_1', 'tov_team_2'],
      dtype='str')

In [6]:
# Only want to look at actual gameplay
stints=stints[stints['start_secs_remaining']!=stints['end_secs_remaining']]

In [7]:
# determine if the stint was in garbage time(gt) or not
stints['is_gt']=np.where((stints['period']==2 ) & (stints['start_secs_remaining']<=300) & ((stints['start_score_1']-stints['start_score_2']).abs()>=20),1,0)

stints.head()

,game_id,stint_number,period,team_1_id,team_2_id,players_team_1,players_team_2,pts_team_1,pts_team_2,poss_team_1,...,end_secs_remaining,fga_team_1,fga_team_2,fta_team_1,fta_team_2,oreb_team_1,oreb_team_2,tov_team_1,tov_team_2,is_gt
0,2682853,1,1,103384,103384,1908702;2134479;2292664;1706626;2319448,2329174;2496768;2160639;2496771;2118863,7,6,8.00,...,2142,7,7,0,0,0,0,1,1,0
8,2682853,9,1,103384,103384,2134479;2292664;2147206;2472149;1688643,2496768;2160639;2496771;2118863;1408733,0,3,0.44,...,2134,0,0,1,1,0,0,0,0,0
10,2682853,11,1,103384,103384,2134479;2292664;2147206;2472149;1688643,2496768;2160639;2118863;1408733;2496767,0,2,1.00,...,2079,1,1,0,0,1,1,1,1,0
12,2682853,13,1,103384,103384,2134479;2147206;2472149;1688643;1706626,2496768;2160639;2118863;1408733;2496767,3,6,2.00,...,1991,2,2,0,0,0,0,0,0,0
16,2682853,17,1,103384,103384,2134479;2147206;2472149;1688643;1706626,1408733;2496767;2161162;1911403;1727511,0,3,1.00,...,1937,1,1,0,0,0,0,0,0,0


In [8]:
# sort lineups for grouping
def split_lineup(lineup):
    ids=lineup.split(';')
    ids_sorted=sorted(ids)
    return ';'.join(ids_sorted)

stints['players_team_1']=stints['players_team_1'].apply(split_lineup)
stints['players_team_2']=stints['players_team_2'].apply(split_lineup)

stints.head()

,game_id,stint_number,period,team_1_id,team_2_id,players_team_1,players_team_2,pts_team_1,pts_team_2,poss_team_1,...,end_secs_remaining,fga_team_1,fga_team_2,fta_team_1,fta_team_2,oreb_team_1,oreb_team_2,tov_team_1,tov_team_2,is_gt
0,2682853,1,1,103384,103384,1706626;1908702;2134479;2292664;2319448,2118863;2160639;2329174;2496768;2496771,7,6,8.00,...,2142,7,7,0,0,0,0,1,1,0
8,2682853,9,1,103384,103384,1688643;2134479;2147206;2292664;2472149,1408733;2118863;2160639;2496768;2496771,0,3,0.44,...,2134,0,0,1,1,0,0,0,0,0
10,2682853,11,1,103384,103384,1688643;2134479;2147206;2292664;2472149,1408733;2118863;2160639;2496767;2496768,0,2,1.00,...,2079,1,1,0,0,1,1,1,1,0
12,2682853,13,1,103384,103384,1688643;1706626;2134479;2147206;2472149,1408733;2118863;2160639;2496767;2496768,3,6,2.00,...,1991,2,2,0,0,0,0,0,0,0
16,2682853,17,1,103384,103384,1688643;1706626;2134479;2147206;2472149,1408733;1727511;1911403;2161162;2496767,0,3,1.00,...,1937,1,1,0,0,0,0,0,0,0


# Stat Calculations

In [9]:
# get stat totals for team 1's lineup
stints['gt_poss']=stints['is_gt']*stints['poss_team_1'] # total garbage time possessions

stints_totals_team1=stints.groupby('players_team_1').agg({'team_1_id':'min','pts_team_1':'sum','pts_team_2':'sum','poss_team_1':'sum','poss_team_2':'sum','fga_team_1':'sum','fga_team_2':'sum','fta_team_1':'sum','fta_team_2':'sum','oreb_team_1':'sum','oreb_team_2':'sum','tov_team_1':'sum','tov_team_2':'sum','gt_poss':'sum'})

stints_totals_team1=stints_totals_team1.reset_index()

stints_totals_team1.columns=stints_totals_team1.columns.str.replace('team_1','for').str.replace('team_2','opp')

stints_totals_team1.head()

stints_totals_team1.sort_values(by='pts_for',ascending=False)


,players_for,for_id,pts_for,pts_opp,poss_for,poss_opp,fga_for,fga_opp,fta_for,fta_opp,oreb_for,oreb_opp,tov_for,tov_opp,gt_poss
30761,1920157;2098978;2478066;2478068;2478069,103296,608,373,503.24,498.00,449,495,171,75,86,102,65,72,9.32
12161,1693954;1911404;1911492;2093260;2334060,103257,533,403,392.40,387.96,377,343,85,59,70,44,48,63,0.00
20448,1890526;1897590;2111602;2505093;2505096,103704,521,360,429.04,421.76,425,368,91,79,79,83,43,102,0.00
5949,1606663;1720487;1890537;1921718;2315141,104161,506,387,419.12,403.24,370,354,123,98,60,68,55,74,2.88
4527,1603371;1953707;2090936;2349987;2540544,104347,422,342,357.72,355.80,311,321,115,70,55,56,51,60,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26037,1908688;2125129;2321242;2506897;2506904,103926,0,4,2.44,2.88,1,2,1,2,0,0,1,0,0.00
26036,1908688;2125129;2321242;2322696;2506897,103926,0,2,2.00,2.00,2,2,0,0,1,0,1,0,0.00
15581,1705477;1925054;1925056;2314352;2319442,104401,0,5,1.88,4.00,2,5,2,0,2,1,1,0,0.00
26034,1908688;1915442;2286440;2321242;2506897,103926,0,0,0.00,0.00,0,0,0,0,0,0,0,0,0.00


In [10]:
# get stat totals for team 2's lineup
stints['gt_poss']=stints['is_gt']*stints['poss_team_2']

stints_totals_team2=stints.groupby('players_team_2').agg({'team_2_id':'min','pts_team_1':'sum','pts_team_2':'sum','poss_team_1':'sum','poss_team_2':'sum','fga_team_1':'sum','fga_team_2':'sum','fta_team_1':'sum','fta_team_2':'sum','oreb_team_1':'sum','oreb_team_2':'sum','tov_team_1':'sum','tov_team_2':'sum','gt_poss':'sum'})

stints_totals_team2=stints_totals_team2.reset_index()

stints_totals_team2.columns=stints_totals_team2.columns.str.replace('team_2','for').str.replace('team_1','opp')

stints_totals_team2.sort_values(by='pts_for',ascending=False)

,players_for,for_id,pts_opp,pts_for,poss_opp,poss_for,fga_opp,fga_for,fta_opp,fta_for,oreb_opp,oreb_for,tov_opp,tov_for,gt_poss
24436,1897500;2111762;2292701;2292702;2292721,103682,463,460,439.96,440.56,393,389,59,74,63,50,83,69,0.00
4441,1603371;1953707;2090936;2349987;2540544,104347,367,447,363.64,371.96,323,335,81,84,53,58,58,58,0.00
26163,1908328;1923950;2125134;2161308;2547382,103300,489,441,441.44,458.40,400,375,101,110,83,54,80,88,0.00
6004,1607446;1909455;1909862;1909863;2301113,103811,363,410,352.12,358.60,330,306,73,65,62,17,52,41,0.00
8800,1681739;1927928;1949395;2342508;2342511,104175,383,392,330.16,328.72,334,292,64,88,63,42,31,40,4.88
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23508,1895187;2143980;2304016;2362609;2535230,103930,1,0,0.44,1.00,0,1,1,0,0,0,0,0,0.00
20039,1737797;1936261;2099012;2567180;2567182,103706,0,0,1.00,0.44,0,0,0,1,0,0,1,0,0.00
34007,1925932;2143519;2144725;2302801;2474952,103269,2,0,1.88,2.00,0,2,2,0,0,0,1,0,0.00
15133,1702071;1926233;1938259;2113991;2126383,104209,1,0,1.88,1.44,1,1,2,1,0,0,0,0,0.00


In [11]:
# combine team 1/team 2 dfs to get total stats

stints_totals=pd.concat([stints_totals_team1,stints_totals_team2])

stints_totals=stints_totals.groupby('players_for').sum().reset_index()


In [12]:
# get team 1 stat totals
totals_team1=stints.groupby('team_1_id').agg({'pts_team_1':'sum','pts_team_2':'sum','poss_team_1':'sum','poss_team_2':'sum','fga_team_1':'sum','fga_team_2':'sum','fta_team_1':'sum','fta_team_2':'sum','oreb_team_1':'sum','oreb_team_2':'sum','tov_team_1':'sum','tov_team_2':'sum'})

totals_team1=totals_team1.reset_index()

totals_team1.columns=totals_team1.columns.str.replace('team_1','for').str.replace('team_2','opp')

totals_team1.head()

totals_team1.sort_values(by='pts_for',ascending=False)

,for_id,pts_for,pts_opp,poss_for,poss_opp,fga_for,fga_opp,fta_for,fta_opp,oreb_for,oreb_opp,tov_for,tov_opp
12,103296,2119,1685,1850.08,1883.36,1608,1687,685,394,344,288,282,311
171,103902,2093,1600,1771.60,1792.56,1513,1617,517,399,261,276,288,273
15,103301,2073,1720,1715.56,1726.16,1518,1537,553,414,249,268,203,275
75,103549,2027,1622,1758.44,1776.92,1567,1577,552,343,343,253,289,302
18,103308,2016,1775,1707.88,1732.84,1483,1458,627,464,303,240,252,310
...,...,...,...,...,...,...,...,...,...,...,...,...,...
51,103454,599,607,605.00,602.84,547,509,153,161,101,109,89,131
270,104230,567,546,590.08,592.60,487,438,159,215,76,70,108,130
61,103504,545,610,586.08,587.84,487,469,183,213,107,89,125,114
69,103531,536,578,599.16,595.44,491,450,166,176,84,64,119,132


In [13]:
# get team 2 stat totals
totals_team2=stints.groupby('team_2_id').agg({'pts_team_1':'sum','pts_team_2':'sum','poss_team_1':'sum','poss_team_2':'sum','fga_team_1':'sum','fga_team_2':'sum','fta_team_1':'sum','fta_team_2':'sum','oreb_team_1':'sum','oreb_team_2':'sum','tov_team_1':'sum','tov_team_2':'sum'})

totals_team2=totals_team2.reset_index()

totals_team2.columns=totals_team2.columns.str.replace('team_2','for').str.replace('team_1','opp')

totals_team2.head()

totals_team2.sort_values(by='pts_for',ascending=False)

,for_id,pts_opp,pts_for,poss_opp,poss_for,fga_opp,fga_for,fta_opp,fta_for,oreb_opp,oreb_for,tov_opp,tov_for
96,103634,1508,1497,1405.88,1408.72,1200,1209,453,414,255,203,261,220
118,103716,1475,1495,1479.76,1486.52,1226,1290,456,408,170,208,221,225
62,103506,1412,1472,1268.24,1252.68,1088,1112,423,301,200,193,193,200
234,104108,1577,1465,1481.00,1483.24,1261,1245,503,446,284,198,282,240
239,104117,1570,1458,1368.68,1376.28,1224,1209,422,362,240,210,199,218
...,...,...,...,...,...,...,...,...,...,...,...,...,...
269,104229,825,705,728.92,722.08,648,627,218,182,120,104,105,119
28,103363,817,701,791.00,794.16,688,708,251,189,120,132,111,135
3,103265,936,681,835.84,837.04,746,660,261,166,138,85,113,188
58,103484,785,676,678.16,663.44,570,565,214,176,93,83,106,104


In [14]:
team_totals=pd.concat([totals_team1,totals_team2])

team_totals=team_totals.groupby('for_id').sum().reset_index()

In [15]:
pt=pd.merge(players,teams,on='team_id',how='left')

In [16]:
# calculate offensive, defensive and net ratings for each team
team_totals['ortg']=(team_totals['pts_for']/team_totals['poss_for'])*100
team_totals['drtg']=(team_totals['pts_opp']/team_totals['poss_opp'])*100
team_totals['net_rating']=(team_totals['ortg']-team_totals['drtg'])

team_totals

teams_map=pt.set_index('team_id')['team_name'].to_dict()
team_totals['team']=team_totals['for_id'].apply(lambda for_id: teams_map.get(for_id,'Unknown') if for_id else 'Unknown')


team_totals.sort_values(by='net_rating',ascending=False).head(15)

,for_id,pts_for,pts_opp,poss_for,poss_opp,fga_for,fga_opp,fta_for,fta_opp,oreb_for,oreb_opp,tov_for,tov_opp,ortg,drtg,net_rating,team
104,103652,2752,2168,2390.12,2414.92,2205,2012,624,568,415,332,324,483,115.140662,89.775231,25.365432,Bulldogs
75,103549,3032,2411,2609.52,2641.36,2326,2347,810,495,493,372,417,448,116.189951,91.278735,24.911215,Blue Devils
171,103902,3041,2451,2630.96,2662.20,2273,2444,737,582,406,444,435,403,115.585186,92.066712,23.518474,Wolverines
90,103614,2628,2106,2270.20,2276.88,2099,2007,687,556,493,316,360,341,115.760726,92.494993,23.265733,Gators
255,104183,2752,2238,2376.32,2385.48,2024,2063,606,698,328,364,413,376,115.809319,93.817596,21.991722,Billikens
114,103704,2690,2100,2403.36,2333.80,2331,1941,544,672,460,365,292,460,111.926636,89.982004,21.944632,Cougars
127,103735,2925,2356,2607.08,2606.28,2344,2126,734,563,454,362,393,592,112.194486,90.397041,21.797445,Cyclones
12,103296,3132,2591,2741.16,2802.16,2406,2503,969,614,506,426,412,455,114.258197,92.464385,21.793813,Wildcats
316,104361,2976,2417,2675.88,2668.44,2447,2227,653,830,485,413,426,486,111.215750,90.577266,20.638483,Huskies
120,103720,2818,2371,2382.40,2413.52,2249,2312,689,408,491,363,320,285,118.284083,98.238258,20.045825,Fighting Illini


# Functions

In [17]:
from itertools import combinations

# function to find every possible lineup coombination
def find_combos(row,size):
    players=row['players_for'].split(';')
    records=[]
    for combo in combinations(sorted(players),size):
        records.append({
            'combo':combo,
            'combo_size':size,
            'points':row['pts_for'],
            'possessions':row['poss_for'],
            'opp_points':row['pts_opp'],
            'opp_poss':row['poss_opp'],
            'gt_poss':row['gt_poss']
        })
    return records

In [18]:
# function to calculate ratings
def calc_ratings(df):
    df['ortg']=(df['points']/df['possessions'])*100
    df['drtg']=(df['opp_points']/df['opp_poss'])*100
    df['net_rating']=df['ortg']-df['drtg']
    df['gt_pct']=df['gt_poss']/df['possessions']
    

In [19]:
# function to get team, player names and team ids
def get_names(df,lookup_df):
    lookup_df['player_id']=lookup_df['player_id'].astype(str)
    names=lookup_df.set_index('player_id')['full_name'].to_dict()
    teams=lookup_df.set_index('player_id')['team_name'].to_dict()
    team_id=lookup_df.set_index('player_id')['team_id'].to_dict()

    df['players_names']=df['combo'].apply(lambda combo: tuple(names.get(pid,'Unknown') for pid in combo))
    df['team']=df['combo'].apply(lambda combo: teams.get(combo[0],'Unknown') if combo else 'Unknown')
    df['team_id']=df['combo'].apply(lambda combo: team_id.get(combo[0],'Unknown') if combo else 'Unknown')
    

In [20]:
# function to get net rating impact for each player/combination
def get_comp_ratings(df):
    team_net_rating_map=team_totals.set_index('for_id')['net_rating']
    team_off_rating_map=team_totals.set_index('for_id')['ortg']
    team_def_rating_map=team_totals.set_index('for_id')['drtg']

    df['net_rating_impact']=df['net_rating']-df['team_id'].map(team_net_rating_map)
    df['off_rating_impact']=df['ortg']-df['team_id'].map(team_off_rating_map)
    df['def_rating_impact']=df['team_id'].map(team_def_rating_map)-df['drtg']
    

# 2-player Combos

In [21]:
# create a dataframe with every 2 player combination
all_rows=[]
for _, row in stints_totals.iterrows():
    all_rows.extend(find_combos(row,2))

expanded=pd.DataFrame(all_rows)

two_combos=(
    expanded.groupby(['combo_size','combo'])
    .agg({'points':'sum','possessions':'sum','opp_points':'sum','opp_poss':'sum','gt_poss':'sum'})
    .reset_index()
)
calc_ratings(two_combos)
get_names(two_combos,pt)
get_comp_ratings(two_combos)

In [22]:
# filter out low possession combos and show most impactful combos
two_combos=two_combos[two_combos['possessions']>1000]

two_combos.sort_values(by='net_rating',ascending=False).head(10)

,combo_size,combo,points,possessions,opp_points,opp_poss,gt_poss,ortg,drtg,net_rating,gt_pct,players_names,team,team_id,net_rating_impact,off_rating_impact,def_rating_impact
24913,2,"(2319989, 2561473)",1213,1007.84,885,1025.00,6.00,120.356406,86.341463,34.014942,0.005953,"(Patrick Ngongba, Cameron Boozer)",Blue Devils,103549,9.103727,4.166455,4.937272
15006,2,"(2098978, 2478069)",1213,1055.80,870,1061.52,10.32,114.889184,81.957947,32.931236,0.009775,"(Motiejus Krivas, Koa Peat)",Wildcats,103296,11.137424,0.630986,10.506438
28644,2,"(2561473, 2561477)",1515,1272.24,1150,1290.16,8.00,119.081305,89.136231,29.945074,0.006288,"(Cameron Boozer, Dame Sarr)",Blue Devils,103549,5.033859,2.891355,2.142504
15005,2,"(2098978, 2478068)",1590,1377.44,1206,1398.76,12.32,115.431525,86.219223,29.212302,0.008944,"(Motiejus Krivas, Brayden Burries)",Wildcats,103296,7.418490,1.173328,6.245162
14854,2,"(2098119, 2106223)",1331,1100.80,1010,1097.92,6.76,120.912064,91.992131,28.919933,0.006141,"(Yaxel Lendeborg, Aday Mara)",Wolverines,103902,5.401459,5.326878,0.074581
24902,2,"(2319985, 2561473)",1954,1620.84,1517,1648.68,5.00,120.554774,92.013004,28.541770,0.003085,"(Isaiah Evans, Cameron Boozer)",Blue Devils,103549,3.630554,4.364823,-0.734269
7623,2,"(1908432, 2535039)",1374,1170.80,1043,1173.84,13.32,117.355654,88.853677,28.501977,0.011377,"(Joshua Jefferson, Killyan Toure)",Cyclones,103735,6.704533,5.161168,1.543364
19961,2,"(2138832, 2561473)",1547,1274.84,1200,1292.36,3.00,121.348561,92.853385,28.495177,0.002353,"(Caleb Foster, Cameron Boozer)",Blue Devils,103549,3.583961,5.158611,-1.574649
26973,2,"(2478068, 2478069)",1678,1429.96,1290,1446.92,26.84,117.345940,89.154895,28.191045,0.018770,"(Brayden Burries, Koa Peat)",Wildcats,103296,6.397233,3.087742,3.309490
8816,2,"(1911404, 2093260)",1373,1078.32,1057,1064.56,5.32,127.327695,99.289847,28.037847,0.004934,"(Fletcher Loyer, Oscar Cluff)",Boilermakers,103257,12.961573,8.352051,4.609521


# 3-player Combos

In [23]:
all_rows=[]
for _, row in stints_totals.iterrows():
    all_rows.extend(find_combos(row,3))

expanded=pd.DataFrame(all_rows)

three_combos=(
    expanded.groupby(['combo_size','combo'])
    .agg({'points':'sum','possessions':'sum','opp_points':'sum','opp_poss':'sum','gt_poss':'sum'})
    .reset_index()
    .sort_values(['combo_size','points'],ascending=[True,False])
)
calc_ratings(three_combos)
get_names(three_combos,pt)
get_comp_ratings(three_combos)

In [24]:
three_combos=three_combos[three_combos['possessions']>800]
three_combos.sort_values(by='net_rating',ascending=False).head(10)

,combo_size,combo,points,possessions,opp_points,opp_poss,gt_poss,ortg,drtg,net_rating,gt_pct,players_names,team,team_id,net_rating_impact,off_rating_impact,def_rating_impact
39765,3,"(1920157, 2098978, 2478069)",1106,950.40,763,948.28,10.32,116.372054,80.461467,35.910587,0.010859,"(Jaden Bradley, Motiejus Krivas, Koa Peat)",Wildcats,103296,14.116774,2.113857,12.002918
55211,3,"(2098978, 2478066, 2478069)",1050,897.64,742,896.24,10.32,116.973397,82.790324,34.183073,0.011497,"(Motiejus Krivas, Ivan Kharchenkov, Koa Peat)",Wildcats,103296,12.389260,2.715200,9.674061
55212,3,"(2098978, 2478068, 2478069)",1059,919.72,752,922.64,10.32,115.143739,81.505246,33.638494,0.011221,"(Motiejus Krivas, Brayden Burries, Koa Peat)",Wildcats,103296,11.844681,0.885542,10.959139
27673,3,"(1897590, 2111602, 2505093)",1021,862.08,715,836.80,0.00,118.434484,85.444551,32.989933,0.000000,"(Milos Uzan, Joseph Tugler, Chris Cenac Jr)",Cougars,103704,11.045301,6.507848,4.537453
39773,3,"(1920157, 2478068, 2478069)",1288,1096.32,951,1104.68,23.84,117.483946,86.088279,31.395667,0.021745,"(Jaden Bradley, Brayden Burries, Koa Peat)",Wildcats,103296,9.601855,3.225749,6.376106
39764,3,"(1920157, 2098978, 2478068)",1305,1123.52,962,1129.76,10.32,116.152805,85.150828,31.001977,0.009185,"(Jaden Bradley, Motiejus Krivas, Brayden Burries)",Wildcats,103296,9.208164,1.894608,7.313556
35084,3,"(1911404, 2093260, 2334060)",1161,881.84,874,867.12,2.32,131.656536,100.793431,30.863105,0.002631,"(Fletcher Loyer, Oscar Cluff, C.J. Cox)",Boilermakers,103257,15.786831,12.680893,3.105938
39771,3,"(1920157, 2478066, 2478069)",1371,1166.00,1020,1171.64,20.96,117.581475,87.057458,30.524017,0.017976,"(Jaden Bradley, Ivan Kharchenkov, Koa Peat)",Wildcats,103296,8.730205,3.323278,5.406927
30866,3,"(1908432, 2099047, 2535039)",1101,937.68,814,932.16,7.44,117.417456,87.324065,30.093391,0.007934,"(Joshua Jefferson, Milan Momcilovic, Killyan Toure)",Cyclones,103735,8.295947,5.222970,3.072976
68437,3,"(2138832, 2319985, 2561473)",1212,992.32,936,1015.80,2.00,122.138020,92.144123,29.993897,0.002015,"(Caleb Foster, Isaiah Evans, Cameron Boozer)",Blue Devils,103549,5.082682,5.948069,-0.865388


# 4-player combos

In [25]:
all_rows=[]
for _, row in stints_totals.iterrows():
    all_rows.extend(find_combos(row,4))

expanded=pd.DataFrame(all_rows)

four_combos=(
    expanded.groupby(['combo_size','combo'])
    .agg({'points':'sum','possessions':'sum','opp_points':'sum','opp_poss':'sum','gt_poss':'sum'})
    .reset_index()
    .sort_values(['combo_size','points'],ascending=[True,False])
)
calc_ratings(four_combos)
get_names(four_combos,pt)
get_comp_ratings(four_combos)


In [26]:
four_combos=four_combos[four_combos['possessions']>600]

four_combos.sort_values(by='net_rating',ascending=False).head(10)

,combo_size,combo,points,possessions,opp_points,opp_poss,gt_poss,ortg,drtg,net_rating,gt_pct,players_names,team,team_id,net_rating_impact,off_rating_impact,def_rating_impact
70080,4,"(1920157, 2098978, 2478066, 2478069)",967,819.32,653,812.64,10.32,118.024703,80.355385,37.669318,0.012596,"(Jaden Bradley, Motiejus Krivas, Ivan Kharchenkov, Koa Peat)",Wildcats,103296,15.875506,3.766506,12.109000
70081,4,"(1920157, 2098978, 2478068, 2478069)",952,814.32,645,811.40,10.32,116.907358,79.492236,37.415123,0.012673,"(Jaden Bradley, Motiejus Krivas, Brayden Burries, Koa Peat)",Wildcats,103296,15.621310,2.649161,12.972149
91990,4,"(2098978, 2478066, 2478068, 2478069)",933,798.16,654,795.88,10.32,116.893856,82.173192,34.720664,0.012930,"(Motiejus Krivas, Ivan Kharchenkov, Brayden Burries, Koa Peat)",Wildcats,103296,12.926851,2.635659,10.291193
107403,4,"(2138832, 2319985, 2319989, 2561473)",758,614.36,564,629.56,1.00,123.380428,89.586378,33.794051,0.001628,"(Caleb Foster, Isaiah Evans, Patrick Ngongba, Cameron Boozer)",Blue Devils,103549,8.882835,7.190478,1.692357
44331,4,"(1890526, 1897590, 2111602, 2505093)",864,735.08,605,710.16,0.00,117.538227,85.192069,32.346158,0.000000,"(Emanuel Sharp, Milos Uzan, Joseph Tugler, Chris Cenac Jr)",Cougars,103704,10.401525,5.611591,4.789934
70087,4,"(1920157, 2478066, 2478068, 2478069)",1126,955.48,821,959.96,20.96,117.846527,85.524397,32.322131,0.021937,"(Jaden Bradley, Ivan Kharchenkov, Brayden Burries, Koa Peat)",Wildcats,103296,10.528318,3.588330,6.939988
63038,4,"(1911404, 1911492, 2093260, 2334060)",1089,814.28,817,805.04,2.32,133.737781,101.485640,32.252140,0.002849,"(Fletcher Loyer, Braden Smith, Oscar Cluff, C.J. Cox)",Boilermakers,103257,17.175866,14.762137,2.413728
70079,4,"(1920157, 2098978, 2478066, 2478068)",1109,951.96,809,953.60,10.32,116.496491,84.836409,31.660082,0.010841,"(Jaden Bradley, Motiejus Krivas, Ivan Kharchenkov, Brayden Burries)",Wildcats,103296,9.866269,2.238294,7.627975
51094,4,"(1897590, 2111602, 2505093, 2505096)",857,718.80,618,697.12,0.00,119.226489,88.650448,30.576041,0.000000,"(Milos Uzan, Joseph Tugler, Chris Cenac Jr, Kingston Flemings)",Cougars,103704,8.631409,7.299853,1.331556
25472,4,"(1693954, 1911404, 2093260, 2334060)",931,704.24,703,689.04,2.32,132.199250,102.026007,30.173243,0.003294,"(Trey Kaufman-Renn, Fletcher Loyer, Oscar Cluff, C.J. Cox)",Boilermakers,103257,15.096968,13.223607,1.873362


# Full Lineup Combos

In [27]:
all_rows=[]
for _, row in stints_totals.iterrows():
    all_rows.extend(find_combos(row,5))

expanded=pd.DataFrame(all_rows)

five_combos=(
    expanded.groupby(['combo_size','combo'])
    .agg({'points':'sum','possessions':'sum','opp_points':'sum','opp_poss':'sum','gt_poss':'sum'})
    .reset_index()
    .sort_values(['combo_size','points'],ascending=[True,False])
)
calc_ratings(five_combos)
get_names(five_combos,pt)
get_comp_ratings(five_combos)

In [28]:
five_combos=five_combos[five_combos['possessions']>300]

five_combos.sort_values(by='net_rating',ascending=False).head(10)

,combo_size,combo,points,possessions,opp_points,opp_poss,gt_poss,ortg,drtg,net_rating,gt_pct,players_names,team,team_id,net_rating_impact,off_rating_impact,def_rating_impact
62038,5,"(2138832, 2319985, 2319989, 2561473, 2561477)",411,328.92,275,332.24,0.00,124.954396,82.771490,42.182906,0.000000,"(Caleb Foster, Isaiah Evans, Patrick Ngongba, Cameron Boozer, Dame Sarr)",Blue Devils,103549,17.271690,8.764446,8.507245
45014,5,"(1920157, 2098978, 2478066, 2478068, 2478069)",850,719.84,565,714.28,10.32,118.081796,79.100633,38.981163,0.014337,"(Jaden Bradley, Motiejus Krivas, Ivan Kharchenkov, Brayden Burries, Koa Peat)",Wildcats,103296,17.187351,3.823599,13.363752
30250,5,"(1890529, 2113314, 2305305, 2349342, 2490633)",389,323.72,273,306.00,0.00,120.165575,89.215686,30.949889,0.000000,"(Jayden Pierre, Brock Harding, David Punch, Liutauras Lelevicius, Xavier Edmonds)",Horned Frogs,104306,22.676934,13.990803,8.686132
17851,5,"(1693954, 1911404, 1911492, 2093260, 2334060)",882,657.88,668,645.28,2.32,134.067003,103.520952,30.546051,0.003526,"(Trey Kaufman-Renn, Fletcher Loyer, Braden Smith, Oscar Cluff, C.J. Cox)",Boilermakers,103257,15.469776,15.091360,0.378417
47183,5,"(1925377, 2112580, 2137468, 2313570, 2549892)",554,486.28,387,463.76,2.00,113.926133,83.448335,30.477798,0.004113,"(Keon Thompson, Kam Burton, Chrishawn Christmas, Jerald Colonel, Lateef Patrick Jr)",Lumberjacks,104159,18.278976,6.276133,12.002843
30128,5,"(1890526, 1897590, 2111602, 2505093, 2505096)",730,614.12,530,593.68,0.00,118.869276,89.273683,29.595594,0.000000,"(Emanuel Sharp, Milos Uzan, Joseph Tugler, Chris Cenac Jr, Kingston Flemings)",Cougars,103704,7.650961,6.942640,0.708321
18795,5,"(1695270, 1910658, 1911503, 1919982, 2281211)",486,412.24,361,406.92,0.00,117.892490,88.715227,29.177263,0.000000,"(Michael Gray Jr., Kenyon Giles, Will Berg, Karon Boyd, TJ Williams)",Shockers,104471,21.113458,12.491043,8.622416
31435,5,"(1890656, 1895929, 1949206, 1949207, 2266043)",392,308.24,328,322.12,1.32,127.173631,101.825407,25.348224,0.004282,"(Shammah Scott, Evan Mahaffey, Amani Lyles, Tavari Johnson, Eric Mahaffey)",Zips,103266,10.408950,10.352000,0.056950
15774,5,"(1688263, 1947760, 2302280, 2533521, 2533591)",394,316.00,307,307.32,0.00,124.683544,99.895874,24.787670,0.000000,"(Trevon Brazile, Malique Ewin, Billy Richmond III, Darius Acuff Jr., Meleek Thomas)",Razorbacks,103301,10.452904,7.431709,3.021195
12133,5,"(1637512, 2126088, 2138845, 2161207, 2169823)",403,338.48,322,340.92,1.00,119.061688,94.450311,24.611377,0.002954,"(Josh Odunowo, Ace Valentine, Jah'Likai King, Jose Roberto Tanchyn, DJ Armstrong)",Retrievers,104364,14.816985,11.835603,2.981382


# Looking into SFU Lumberjacks Conference

In [29]:
team_totals=pd.merge(team_totals,teams,left_on='for_id',right_on='team_id',how='left')


In [30]:
team_totals[team_totals['conference_id']==72].sort_values(by='net_rating',ascending=False)

,for_id,pts_for,pts_opp,poss_for,poss_opp,fga_for,fga_opp,fta_for,fta_opp,oreb_for,...,tov_for,tov_opp,ortg,drtg,net_rating,team,team_id,team_market,team_name,conference_id
164,103882,2248,1984,2124.52,2123.16,1931,1706,634,623,393,...,304,517,105.812136,93.445619,12.366518,Cowboys,103882,McNeese,Cowboys,72
248,104159,2153,1907,2000.00,1997.88,1856,1743,527,478,371,...,282,355,107.650000,95.451178,12.198822,Lumberjacks,104159,SFA,Lumberjacks,72
331,104381,2108,1967,1997.24,2007.52,1723,1767,422,610,271,...,359,325,105.545653,97.981589,7.564064,Vaqueros,104381,UTRGV,Vaqueros,72
1,103258,1851,1883,1933.24,1948.28,1617,1559,575,665,318,...,379,371,95.746002,96.649352,-0.903351,Islanders,103258,A&M-Corpus Christi,Islanders,72
142,103791,1825,1841,1865.24,1856.16,1712,1538,424,619,376,...,342,325,97.842637,99.183260,-1.340623,Cardinals,103791,Lamar University,Cardinals,72
198,103986,2406,2479,2387.96,2394.08,1950,2132,838,764,420,...,487,380,100.755457,103.547083,-2.791626,Privateers,103986,LSU New Orleans,Privateers,72
200,103992,2141,2217,2141.40,2123.72,1823,1740,585,616,316,...,377,440,99.981321,104.392293,-4.410972,Colonels,103992,Nicholls,Colonels,72
122,103724,1929,1997,1863.84,1835.96,1643,1579,492,589,323,...,325,287,103.496008,108.771433,-5.275425,Cardinals,103724,UIW,Cardinals,72
115,103705,1870,2068,1954.88,1967.20,1645,1724,558,483,308,...,370,353,95.658046,105.124034,-9.465989,Huskies,103705,Houston Christian,Huskies,72
275,104237,1756,1953,1920.12,1903.56,1600,1574,581,578,295,...,358,366,91.452618,102.597239,-11.144621,Lions,104237,Southeastern La.,Lions,72
